In [1]:
# CONFIG
import os, json, time, random, subprocess
from pathlib import Path
import numpy as np
import anthropic
import warnings
import faiss
from sentence_transformers import SentenceTransformer
import statistics as stats
from openai import OpenAI
import tiktoken
import yaml as _yaml
import os, json, time, random, subprocess
from pathlib import Path
import numpy as np

SEED         = 42
CHUNK_TOKENS = 256
K            = 5
N_QUESTIONS  = 50
FORMATS      = ["json", "toon", "yaml"]          # baseline first
CELL_NAMES   = {"json": "json_fixed_256", "toon": "toon_fixed_256", "yaml": "yaml_fixed_256"}

GPT_MODEL    = "gpt-4o-mini"
CLAUDE_MODEL = "claude-haiku-4-5-20251001"
CTX_MODEL    = "claude-haiku-4-5-20251001"
JUDGE_MODEL  = "claude-haiku-4-5-20251001"
EMBED_MODEL  = "jinaai/jina-embeddings-v2-base-en"
GEN_PARAMS   = dict(temperature=0, top_p=1, max_tokens=512)

for d in ("data", "cache", "results/hotpotqa"):
    Path(d).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "NO-GIT"
COMMIT = git_commit()

assert os.environ.get("OPENAI_API_KEY"),    "OPENAI_API_KEY not set"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
print(f"Config OK — commit {COMMIT} | seed {SEED} | {len(FORMATS)} cells | {N_QUESTIONS} questions")

C:\Users\enasz\Desktop\RAG-Context\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config OK — commit 2c6a976 | seed 42 | 3 cells | 50 questions


## Cell 2 — Import modules

In [2]:
# PIPELINE CODE
_enc = tiktoken.get_encoding("cl100k_base")

# ---------- chunking (fixed-size, token windows) ----------
def fixed_chunks(text, size_tokens=CHUNK_TOKENS):
    toks = _enc.encode(text)
    return [_enc.decode(toks[i:i + size_tokens]) for i in range(0, len(toks), size_tokens)]

# ---------- token accounting ----------
def count_gpt(text):
    return len(_enc.encode(text))

FIELD_ORDER = ["doc_id", "title", "chunk_id", "context", "text"]

def serialize(record: dict, fmt: str) -> str:
    if fmt == "json":
        return to_json(record)
    if fmt == "toon":
        return to_toon(record)
    if fmt == "yaml":
        return to_yaml(record)
    raise ValueError(f"unknown format: {fmt}")

# ---- JSON (baseline) --------------------------------------------------
def to_json(record: dict) -> str:
    ordered = {k: record[k] for k in FIELD_ORDER if k in record}
    return json.dumps(ordered, ensure_ascii=False, separators=(",", ":"))

# ---- TOON (flat records: 'key: value' lines) --------------------------
def _escape_toon(v: str) -> str:
    return v.replace("\\", "\\\\").replace("\n", "\\n")

def _unescape_toon(v: str) -> str:
    out, i = [], 0
    while i < len(v):
        if v[i] == "\\" and i + 1 < len(v):
            out.append("\n" if v[i + 1] == "n" else v[i + 1]); i += 2
        else:
            out.append(v[i]); i += 1
    return "".join(out)

def to_toon(record: dict) -> str:
    return "\n".join(f"{k}: {_escape_toon(str(record[k]))}"
                     for k in FIELD_ORDER if k in record)

def from_toon(text: str) -> dict:
    out = {}
    for line in text.split("\n"):
        if ": " in line:
            k, v = line.split(": ", 1)
            out[k] = _unescape_toon(v)
    return out

# ---- YAML -------------------------------------------------------------
def to_yaml(record: dict) -> str:
    import yaml  # lazy: only needed when yaml cells run
    ordered = {k: record[k] for k in FIELD_ORDER if k in record}
    return yaml.safe_dump(ordered, sort_keys=False,
                          allow_unicode=True, width=10**9).strip()

# ---- lossless validation (required before indexing) --------------------
def roundtrip_ok(record: dict) -> bool:
    ordered = {k: str(record[k]) for k in FIELD_ORDER if k in record}
    return from_toon(to_toon(record)) == ordered

SERIALIZERS = {"json": to_json, "toon": to_toon, "yaml": to_yaml}
print("Pipeline code loaded — thesis serialise functions inline (FIELD_ORDER:", FIELD_ORDER, ")")

Pipeline code loaded — thesis serialise functions inline (FIELD_ORDER: ['doc_id', 'title', 'chunk_id', 'context', 'text'] )


## Dataset

In [3]:
CORPUS_F, QUEST_F = Path("data/corpus.jsonl"), Path("data/questions.jsonl")

if CORPUS_F.exists() and QUEST_F.exists():
    corpus    = [json.loads(l) for l in CORPUS_F.open(encoding="utf-8")]
    questions = [json.loads(l) for l in QUEST_F.open(encoding="utf-8")]
    print(f"Checkpoint: {len(corpus)} docs, {len(questions)} questions.")
else:
    from datasets import load_dataset
    ds  = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation", trust_remote_code=True)
    idx = random.Random(SEED).sample(range(len(ds)), N_QUESTIONS)
    questions, corpus, seen = [], [], set()
    for i in idx:
        ex = ds[i]
        questions.append({"qid": ex["id"], "question": ex["question"], "answer": ex["answer"],
                          "gold_titles": sorted(set(ex["supporting_facts"]["title"]))})
        for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
            if title not in seen:
                seen.add(title)
                corpus.append({"doc_id": f"doc_{len(corpus):04d}", "title": title,
                               "text": " ".join(sents)})
    with CORPUS_F.open("w", encoding="utf-8") as f:
        for d in corpus: f.write(json.dumps(d, ensure_ascii=False) + "\n")
    with QUEST_F.open("w", encoding="utf-8") as f:
        for q in questions: f.write(json.dumps(q, ensure_ascii=False) + "\n")
    print(f"Built {len(corpus)} docs from {len(questions)} questions' pooled contexts.")

Checkpoint: 498 docs, 50 questions.


In [4]:
# CHUNKING
chunks = []
for doc in corpus:
    for j, piece in enumerate(fixed_chunks(doc["text"], CHUNK_TOKENS)):
        chunks.append({"chunk_id": f'{doc["doc_id"]}_c{j:02d}', "doc_id": doc["doc_id"],
                       "title": doc["title"], "text": piece})
raw_chunk_tokens = sum(count_gpt(c["text"]) for c in chunks)
print(f"{len(chunks)} chunks from {len(corpus)} docs | raw chunk tokens: {raw_chunk_tokens:,}")

525 chunks from 498 docs | raw chunk tokens: 63,713


## Cell 5 — Contextualization

In [5]:
# CONTEXTUALIZATION
_ant = anthropic.Anthropic()
SNIP_F, snippets = Path("cache/snippets.jsonl"), {}
if SNIP_F.exists():
    for l in SNIP_F.open(encoding="utf-8"):
        r = json.loads(l); snippets[r["chunk_id"]] = r["snippet"]
    print(f"Checkpoint: {len(snippets)} snippets cached.")

doc_text = {d["doc_id"]: d["text"] for d in corpus}
todo = [c for c in chunks if c["chunk_id"] not in snippets]
print(f"{len(todo)} snippets to generate ...")

PROMPT = (
    "Here is a chunk from the document above:\n<chunk>\n{chunk}\n</chunk>\n"
    "Write a short (1-2 sentence) context that situates this chunk within "
    "the overall document, to improve search retrieval of the chunk. "
    "Answer with only the context, nothing else."
)

with SNIP_F.open("a", encoding="utf-8") as f:
    for n, c in enumerate(todo, 1):
        m = _ant.messages.create(model=CTX_MODEL, max_tokens=150, temperature=0,
            system=[{"type": "text",
                     "text": f"<document>\n{doc_text[c['doc_id']]}\n</document>",
                     "cache_control": {"type": "ephemeral"}}],
            messages=[{"role": "user", "content": PROMPT.format(chunk=c["text"])}])
        snippets[c["chunk_id"]] = m.content[0].text.strip()
        f.write(json.dumps({"chunk_id": c["chunk_id"], "snippet": snippets[c["chunk_id"]]},
                           ensure_ascii=False) + "\n"); f.flush()
        if n % 50 == 0: print(f"  {n}/{len(todo)}")

snippet_tokens = sum(count_gpt(s) for s in snippets.values())
ctx_overhead = 100 * snippet_tokens / raw_chunk_tokens
print(f"{len(snippets)} snippets | contextualization overhead = {ctx_overhead:.1f}%")

525 snippets to generate ...
  50/525
  100/525
  150/525
  200/525
  250/525
  300/525
  350/525
  400/525
  450/525
  500/525
525 snippets | contextualization overhead = 35.2%


## Cell 6 — Serialization × 3 (the independent variable)

In [6]:
# SERIALIZATION
serialized = {f: [] for f in FORMATS}
tok_index  = {f: 0 for f in FORMATS}
rt_fail = 0

for c in chunks:
    record = {"doc_id": c["doc_id"], "title": c["title"], "chunk_id": c["chunk_id"],
              "context": snippets[c["chunk_id"]], "text": c["text"]}
    if not roundtrip_ok(record): rt_fail += 1
    for fmt in FORMATS:
        s = SERIALIZERS[fmt](record)
        serialized[fmt].append(s)
        tok_index[fmt] += count_gpt(s)

assert rt_fail == 0, f"{rt_fail} round-trip failures — stop and fix serialise.py"
base = tok_index["json"]
for fmt in FORMATS:
    sav = "" if fmt == "json" else f' | saving vs baseline={100*(1-tok_index[fmt]/base):.1f}%'
    print(f'[{fmt}] indexing tokens={tok_index[fmt]:,}{sav}')

[json] indexing tokens=103,276
[toon] indexing tokens=101,728 | saving vs baseline=1.5%
[yaml] indexing tokens=102,077 | saving vs baseline=1.2%


## Cell 7 — Embed + FAISS, one index per format

In [7]:
# EMBED + FAISS
embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True)

indexes = {}
for fmt in FORMATS:
    ef = Path(f"cache/emb_{fmt}.npy")
    if ef.exists():
        emb = np.load(ef); print(f"[{fmt}] embeddings from checkpoint {emb.shape}")
    else:
        emb = embedder.encode(serialized[fmt], batch_size=16, show_progress_bar=True,
                              convert_to_numpy=True)
        np.save(ef, emb)
    emb = emb.astype("float32"); faiss.normalize_L2(emb)
    ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb)
    indexes[fmt] = ix
    print(f"[{fmt}] FAISS: {ix.ntotal} vectors")

Batches: 100%|██████████| 33/33 [01:42<00:00,  3.11s/it]


[json] FAISS: 525 vectors


Batches: 100%|██████████| 33/33 [01:27<00:00,  2.66s/it]


[toon] FAISS: 525 vectors


Batches: 100%|██████████| 33/33 [01:28<00:00,  2.70s/it]

[yaml] FAISS: 525 vectors


## Cell 8 — Retrieval + per-question IR metrics

In [8]:
# RETRIEVAL
q_emb = embedder.encode([q["question"] for q in questions],
                        convert_to_numpy=True).astype("float32")
faiss.normalize_L2(q_emb)

retrieved, ir_per_q = {f: {} for f in FORMATS}, {f: {} for f in FORMATS}
for fmt in FORMATS:
    _, I = indexes[fmt].search(q_emb, K)
    for qi, q in enumerate(questions):
        idxs = I[qi].tolist()
        retrieved[fmt][q["qid"]] = idxs
        titles = [chunks[i]["title"] for i in idxs]
        gold = set(q["gold_titles"])
        r_at5 = len(gold & set(titles)) / len(gold)                       # partial recall
        rr = 0.0
        for rank, t in enumerate(titles):
            if t in gold: rr = 1.0 / (rank + 1); break
        ir_per_q[fmt][q["qid"]] = {"r5": r_at5, "rr": rr}
    print(f'[{fmt}] Recall@{K}={np.mean([v["r5"] for v in ir_per_q[fmt].values()]):.2f}  '
          f'MRR={np.mean([v["rr"] for v in ir_per_q[fmt].values()]):.4f}')

[json] Recall@5=0.88  MRR=0.9733
[toon] Recall@5=0.87  MRR=0.9633
[yaml] Recall@5=0.87  MRR=0.9733


## Cell 9 — Generation: GPT + Claude × 3 formats

In [9]:
# GENERATION
_oai = OpenAI()

GEN_PARAMS_OAI  = dict(temperature=0, top_p=1, max_tokens=512) 
GEN_PARAMS_ANTH = dict(temperature=0, max_tokens=512)          

GEN_F, answers = Path("cache/answers.jsonl"), {}
if GEN_F.exists():
    for l in GEN_F.open(encoding="utf-8"):
        r = json.loads(l); answers[(r["fmt"], r["gen"], r["qid"])] = r
    print(f"Checkpoint: {len(answers)} answers cached.")

SYS = ("Answer the question using ONLY the provided context records. "
       "If the context is insufficient, say so. Be concise.")

def build_context(fmt, qid):
    return "\n\n".join(serialized[fmt][i] for i in retrieved[fmt][qid])

with GEN_F.open("a", encoding="utf-8") as f:
    for fmt in FORMATS:
        done = 0
        for q in questions:
            ctx = build_context(fmt, q["qid"])
            prompt = f"Context records:\n{ctx}\n\nQuestion: {q['question']}"
            for gen in ("gpt", "claude"):
                key = (fmt, gen, q["qid"])
                if key in answers: continue
                t0 = time.time()
                if gen == "gpt":
                    r = _oai.chat.completions.create(model=GPT_MODEL, seed=SEED, **GEN_PARAMS_OAI,
                        messages=[{"role": "system", "content": SYS},
                                  {"role": "user", "content": prompt}])
                    ans = r.choices[0].message.content
                else:
                    r = _ant.messages.create(model=CLAUDE_MODEL, system=SYS, **GEN_PARAMS_ANTH,
                        messages=[{"role": "user", "content": prompt}])
                    ans = r.content[0].text
                rec = {"fmt": fmt, "gen": gen, "qid": q["qid"], "answer": ans,
                       "tok_ctx": count_gpt(ctx), "latency_s": round(time.time() - t0, 2)}
                answers[key] = rec
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush(); done += 1
        print(f"[{fmt}] generation done ({done} new calls)")

Checkpoint: 300 answers cached.
[json] generation done (0 new calls)
[toon] generation done (0 new calls)
[yaml] generation done (0 new calls)


## Cell 10 — Faithfulness judge

In [10]:
# FAITHFULNESS
JUD_F, judgments = Path("cache/judgments.jsonl"), {}
if JUD_F.exists():
    for l in JUD_F.open(encoding="utf-8"):
        r = json.loads(l); judgments[(r["fmt"], r["gen"], r["qid"])] = r["score"]
    print(f"Checkpoint: {len(judgments)} judgments cached.")

JP = ("Context:\n{ctx}\n\nAnswer:\n{ans}\n\n"
      "What fraction of the factual claims in the Answer are directly supported by the "
      "Context? Respond with ONLY a number between 0 and 1.")

with JUD_F.open("a", encoding="utf-8") as f:
    for key, rec in answers.items():
        if key in judgments: continue
        fmt, gen, qid = key
        r = _ant.messages.create(model=JUDGE_MODEL, max_tokens=10, temperature=0,
            messages=[{"role": "user", "content": JP.format(ctx=build_context(fmt, qid),
                                                            ans=rec["answer"])}])
        try:
            score = max(0.0, min(1.0, float(r.content[0].text.strip().split()[0])))
        except ValueError:
            score = float("nan")
        judgments[key] = score
        f.write(json.dumps({"fmt": fmt, "gen": gen, "qid": qid, "score": score}) + "\n")
        f.flush()
print(f"Judging complete: {len(judgments)} scores.")

Checkpoint: 300 judgments cached.
Judging complete: 300 scores.


## Cell 11 — Per-question lines, result block, results JSON with embedded commit

In [11]:
# RESULTS
for fmt in FORMATS:
    print(f"\nPer-question ({fmt}):")
    for q in questions:
        m = ir_per_q[fmt][q["qid"]]
        a = answers[(fmt, "gpt", q["qid"])]
        print(f'{q["qid"][:8]}  R@5={m["r5"]:.2f} MRR={m["rr"]:.2f} '
              f'ctx={a["tok_ctx"]}t {a["latency_s"]:.1f}s')

results = {}
for fmt in FORMATS:
    fa = {g: stats.mean([judgments[(fmt, g, q["qid"])] for q in questions
                         if not np.isnan(judgments[(fmt, g, q["qid"])])])
          for g in ("gpt", "claude")}
    cell = {
        "cell": CELL_NAMES[fmt], "commit": COMMIT, "corpus": "hotpotqa",
        "n_docs": len(corpus), "n_questions": len(questions), "n_chunks": len(chunks),
        "recall_at_5": round(float(np.mean([v["r5"] for v in ir_per_q[fmt].values()])), 4),
        "mrr":         round(float(np.mean([v["rr"] for v in ir_per_q[fmt].values()])), 4),
        "faith_gpt":    round(fa["gpt"], 4), "faith_claude": round(fa["claude"], 4),
        "ctx_tokens_per_query": round(stats.mean(
            [answers[(fmt, "gpt", q["qid"])]["tok_ctx"] for q in questions]), 1),
        "indexing_tokens": tok_index[fmt],
        "contextualization_overhead_pct": round(ctx_overhead, 1) if fmt == "json" else None,
        "token_saving_vs_baseline_pct":
            None if fmt == "json" else round(100 * (1 - tok_index[fmt] / tok_index["json"]), 1),
    }
    results[fmt] = cell
    Path(f'results/hotpotqa/{CELL_NAMES[fmt]}.json').write_text(json.dumps(cell, indent=2))

b = results["json"]
print(f"""
=============== MID-REVIEW SECTION - RESULT BLOCK ================
The baseline cell (JSON x fixed-size, 256 tokens) runs end-to-end on a hotpotqa subset ({b["n_docs"]} documents, {b["n_questions"]} questions, {b["n_chunks"]} chunks). Commit {COMMIT}.
Baseline: Recall@5={b["recall_at_5"]:.2f}  MRR={b["mrr"]:.4f}  faith(GPT)={b["faith_gpt"]:.4f}  faith(Claude)={b["faith_claude"]:.4f}
          ctx tokens/query={b["ctx_tokens_per_query"]}  contextualization overhead={b["contextualization_overhead_pct"]}%  indexing tokens={b["indexing_tokens"]}""")
for fmt in FORMATS[1:]:
    c = results[fmt]
    print(f"""
{CELL_NAMES[fmt]}: Recall@5={c["recall_at_5"]:.2f}  MRR={c["mrr"]:.4f}  faith(GPT)={c["faith_gpt"]:.4f}  faith(Claude)={c["faith_claude"]:.4f}
          ctx tokens/query={c["ctx_tokens_per_query"]}  indexing tokens={c["indexing_tokens"]}  token saving vs baseline={c["token_saving_vs_baseline_pct"]}%""")
print("=" * 66)
if COMMIT == "NO-GIT":
    print("WARNING: not a git repo — commit the repo and rerun Cell 11 only.")


Per-question (json):
5ae143ed  R@5=1.00 MRR=1.00 ctx=910t 2.8s
5abc1970  R@5=0.50 MRR=1.00 ctx=1129t 1.6s
5ac3e0f7  R@5=1.00 MRR=1.00 ctx=896t 0.9s
5ae51865  R@5=1.00 MRR=1.00 ctx=819t 1.4s
5ab985eb  R@5=1.00 MRR=1.00 ctx=1081t 0.7s
5a8753d9  R@5=1.00 MRR=1.00 ctx=880t 1.0s
5ab97d0a  R@5=1.00 MRR=1.00 ctx=1116t 1.6s
5adef1b3  R@5=0.50 MRR=1.00 ctx=712t 0.8s
5a73b558  R@5=1.00 MRR=1.00 ctx=1035t 0.7s
5ae80919  R@5=1.00 MRR=1.00 ctx=787t 1.4s
5ae36160  R@5=0.50 MRR=1.00 ctx=1064t 0.8s
5ac3a76e  R@5=1.00 MRR=1.00 ctx=951t 0.9s
5a7557d7  R@5=1.00 MRR=1.00 ctx=1199t 0.9s
5a8d12ca  R@5=1.00 MRR=1.00 ctx=885t 0.7s
5ab7f6d3  R@5=1.00 MRR=1.00 ctx=794t 0.7s
5ac3d913  R@5=1.00 MRR=1.00 ctx=655t 0.8s
5ae12911  R@5=1.00 MRR=1.00 ctx=909t 0.9s
5a86769c  R@5=1.00 MRR=1.00 ctx=1000t 1.2s
5ab31864  R@5=1.00 MRR=1.00 ctx=1292t 0.8s
5a7b1023  R@5=1.00 MRR=1.00 ctx=1066t 2.0s
5ac19f40  R@5=1.00 MRR=1.00 ctx=878t 0.7s
5ae382ee  R@5=0.50 MRR=1.00 ctx=1385t 0.9s
5adcdf45  R@5=1.00 MRR=1.00 ctx=785t 0.8s
5a

## Cell 12 — Commit + tag (PowerShell)
```powershell
git add -A
git commit -m "Pilot: JSON/TOON/YAML x fixed x 256, 50q pooled HotpotQA (notebook, Restart & Run All)"
git tag pilot-midreview-nb
git rev-parse --short HEAD
```